# Time-varying variance and extreme-day probabilities

A bank's share price is a different object in different years. JPMorgan Chase
returned an average of fifteen basis points a day in 2019 with a daily
standard deviation of 1.2 per cent; in 2020 the standard deviation was 3.4 per
cent and the average was slightly negative. Everyone who works with such a
series knows the volatility moves. The usual response is a rolling standard
deviation: recompute over the last sixty days, and let the number track the
regime.

That answers one question about the distribution and silently assumes the
answer to another. A standard deviation fixes how far a typical day lands from
the centre. It does not fix how often a day lands very far away — that depends
on the shape of the tail, and nothing in a rolling standard deviation
estimates shape. If the tail weight is itself moving, then a series can be
given the right variance for every day of its history and still be told the
wrong thing about its worst days.

This case study fits the whole distribution of one stock's daily returns as a
function of calendar time: a location, a scale and a tail index, each its own
smooth curve, estimated jointly by `gamboostLSS`
{cite:t}`hofner2014gamboostlss` on 1,920 trading days. The question is whether
the third curve earns its place — whether tail weight moves enough,
independently of volatility, to change what the model says a bad day costs.

## Data Preparation and Initial Statistics

This block initializes the data processing pipeline.
It performs the following tasks:
1.  **Data Loading:** Loads the daily return data for JPM from parquet format.
2.  **Feature Engineering:** Calculates two key variables for the regression:
    *   `ret_pct`: The daily total return in percentage points (the dependent variable, $y$).
    *   `t`: The time variable, representing calendar time in years since the start of the sample (the independent covariate).
3.  **Summary Statistics:** Calculates and prints basic descriptive statistics for the entire sample, including mean, standard deviation, and the excess kurtosis (a measure of tail thickness).


In [ ]:
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
import polars as pl
from scipy import stats

import mboostrs as mb


def data_path(name):
    """Resolve book data under MyST and repository-root execution."""
    for directory in (Path("../data"), Path("data"), Path("docs/book/data")):
        candidate = directory / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(name)


TICKER = "JPM"
EXTREME = 5.0  # percent, the size of move the study prices

panel = (
    pl.read_parquet(data_path("sp500_daily.parquet"))
    .filter(pl.col("ticker") == TICKER)
    .sort("date")
    .with_columns(
        # Daily total return in percentage points. The percent scale matters
        # numerically: the location gradient divides by sigma squared, and at
        # decimal-fraction scale fixed-size boosting steps overshoot.
        (pl.col("ret") * 100).alias("ret_pct"),
        # Calendar time in years since the start of the sample, the single
        # covariate every equation reads.
        ((pl.col("date").cast(pl.Date) - pl.date(2019, 1, 1)).dt.total_days()
         / 365.25).cast(pl.Float64).alias("t"),
    )
    .select("date", "t", "ret_pct")
)

data = panel.to_pandas()
security = "JPMorgan Chase"
n_days = panel.height
start, end = panel["date"].min().date(), panel["date"].max().date()
r = panel["ret_pct"].to_numpy()
sample_sd = float(r.std())
sample_kurtosis = float(((r - r.mean()) ** 4).mean() / r.std() ** 4 - 3)
print(f"{security} ({TICKER}): {n_days} trading days, {start} to {end}")
print(f"daily total return: mean {r.mean():+.3f}%, sd {sample_sd:.3f}%, "
      f"range {r.min():.1f}% to {r.max():.1f}%")
print(f"excess kurtosis {sample_kurtosis:.1f} (a normal law has zero)")

## The series

The extract is built by `tools/curate_sp500_daily.py` from Yahoo Finance
adjusted closes, so the log difference is the total return including
dividends. One ticker is pulled out of it, leaving a plain time series: one
return per trading day, and calendar time as the only predictor.

In [ ]:
display(panel)

In [ ]:
by_year = (
    panel.with_columns(pl.col("date").dt.year().alias("year"))
    .group_by("year")
    .agg(pl.len().alias("days"),
         (pl.col("ret_pct").mean() * 252).round(1).alias("annual return (pp)"),
         (pl.col("ret_pct").std() * np.sqrt(252)).round(0).alias("annualised sd (%)"),
         (pl.col("ret_pct").abs() > EXTREME).sum().alias("days beyond 5%"))
    .sort("year")
)
display(by_year)

n_extreme = int((np.abs(r) > EXTREME).sum())
worst = panel.sort("ret_pct").head(1)
print(f"{n_extreme} of {n_days} days moved more than {EXTREME:.0f}% "
      f"({n_extreme / n_days:.2%}); the worst was "
      f"{float(worst['ret_pct'][0]):.1f}% on {worst['date'][0].date()}")

points = alt.Chart(panel).mark_circle(size=7, opacity=0.45).encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("ret_pct:Q", title="Daily total return (%)"),
    tooltip=[alt.Tooltip("date:T"), alt.Tooltip("ret_pct:Q", format="+.2f")],
)
points.properties(title=f"{security} daily returns", width=560, height=260)

## Annual Summary Statistics and Visualization

This section summarizes the historical data by year. It calculates and displays:
1.  **Annual Metrics:** The total number of trading days, annualized mean return, annualized standard deviation, and the count of days with extreme returns (over 5%).
2.  **Visualization:** Plots the raw daily returns over time to give a visual overview of the entire sample period.


The picture is the familiar one for a large bank: a quiet 2019, a violent
first half of 2020, and a series of smaller disturbances since. Two features
of it govern what follows. Volatility plainly moves, so any model with one
variance for the whole sample is wrong. And the extremes are far too frequent
for a normal law — {eval}`n_extreme:d` days beyond five per cent against the
handful a Gaussian with this standard deviation would allow.

### Rolling standard deviation as a scale estimate

The rolling standard deviation estimates local scale as the standard
deviation of the last sixty returns. It follows changes in volatility, but
it does not estimate how frequently returns fall far from the centre.

In [ ]:
WINDOW = 60
rolling = panel.with_columns(
    pl.col("ret_pct").rolling_std(WINDOW).alias("rolling_sd")
).drop_nulls("rolling_sd")

roll_chart = alt.Chart(rolling).mark_line(color="#4e79a7").encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("rolling_sd:Q", title=f"Standard deviation of last {WINDOW} days (%)"),
    tooltip=[alt.Tooltip("date:T"), alt.Tooltip("rolling_sd:Q", format=".2f")],
)
display(roll_chart.properties(
    title=f"Trailing {WINDOW}-day standard deviation", width=560, height=240))

roll = rolling["rolling_sd"].to_numpy()
print(f"trailing standard deviation runs from {roll.min():.2f}% to "
      f"{roll.max():.2f}% per day")

# What that number implies about an extreme day, if returns were normal.
gauss_p = 2 * stats.norm.sf(EXTREME / roll)
print(f"if returns were normal with that standard deviation, the chance of a "
      f"{EXTREME:.0f}% day would run from {gauss_p.min():.2e} to {gauss_p.max():.2%}")
print(f"expected count of {EXTREME:.0f}% days over the sample on that basis: "
      f"{gauss_p.sum():.0f}; observed: {n_extreme:d}")

## Analyzing Rolling Volatility (Standard Deviation)

This block implements the typical "first pass" analysis:
1.  **Rolling Window:** Calculates the standard deviation (`rolling_sd`) over a fixed historical window (60 days).
2.  **Normal Assumption:** It then uses this local standard deviation and *assumes* a Normal distribution to calculate the expected probability of a 5% move.
3.  **Goal:** This demonstrates what the standard, limited approach can estimate, which serves as a baseline to show its shortcomings compared to the full Student-t model.


The trailing standard deviation tracks the regime, and taken with a normal law
it accounts for {eval}`gauss_p.sum():.0f` days beyond five per cent against
the {eval}`n_extreme:d` that occurred. The shortfall is not a detail of the
window length. It is the shape assumption: a normal law reaches five per cent
only by making the typical day large, so the only way it can produce enough
extreme days is to overstate the ordinary ones.

A distributional model must estimate both the scale around the centre and
the rate at which tail probabilities decrease.

## Three curves, one fit

A Student-t distribution separates exactly those two. Its density is

$$f(y) \propto \frac{1}{\sigma}\left(1 + \frac{1}{\nu}\left(\frac{y-\mu}{\sigma}\right)^{2}\right)^{-(\nu+1)/2},$$

with three parameters that do three different jobs: $\mu$ locates the centre,
$\sigma$ sets the scale, and $\nu$ — the degrees of freedom — sets how heavy
the tail is. Small $\nu$ means extremes are common; as $\nu$ grows the
distribution approaches a normal law.

$\sigma$ is the **scale, not the standard deviation**. The two differ by a
factor that depends on $\nu$:

$$\operatorname{sd} = \sigma\sqrt{\frac{\nu}{\nu-2}}, \qquad \nu > 2,$$

which is 1.3 at $\nu = 5$ and 2.1 at $\nu = 2.6$, and infinite at $\nu \le 2$.
Reading $\sigma$ as a volatility would therefore compare two periods on a
scale whose meaning changed between them, and every quantity below is
converted before it is compared.

`gamboostLSS` fits all three parameters at once, each with its own equation
and its own additive predictor {cite:t}`hofner2014gamboostlss`. Here every
equation gets the same single candidate — a penalised spline in calendar time
— so there is no variable selection in this study and nothing is competing.
The fit estimates three curves:

* $\mu(t)$, identity link — the drift;
* $\sigma(t)$, log link — the scale;
* $\nu(t)$, log link — the tail index.

The spline is `bbs(t, df=4)`, a B-spline basis with a second-difference
penalty tuned to four degrees of freedom {cite:t}`eilers1996flexible`. That
number caps how much structure any single boosting step can add; the total
flexibility of the fitted curve is set by the number of iterations, which is
the next section's problem.

In [ ]:
DF_SPLINE = 4
FORMS = {name: f"ret_pct ~ bbs(t, df={DF_SPLINE})"
         for name in ("mu", "sigma", "df")}

gauss = mb.gamboostLSS({k: FORMS[k] for k in ("mu", "sigma")}, data=data,
                       families=mb.GaussianLSS(),
                       control=mb.boost_control(1000, 0.1))
student = mb.gamboostLSS(FORMS, data=data, families=mb.StudentTLSS(),
                         control=mb.boost_control(1000, 0.1))


def residual_kurtosis(fit, mstop):
    """Excess kurtosis of the randomised quantile residuals.

    Under a correctly specified fit these residuals are standard normal
    whatever the family, so this reads zero when the shape is right.
    """
    q = np.asarray(mb.quantile_residuals(fit, mstop=mstop, random_state=7))
    return float(((q - q.mean()) ** 4).mean() / q.std() ** 4 - 3)


gauss_kurtosis = residual_kurtosis(gauss, 1000)
student_kurtosis = residual_kurtosis(student, 1000)
display(pl.DataFrame({
    "fit": ["Gaussian, sigma(t)", "Student-t, sigma(t) and nu(t)"],
    "residual excess kurtosis": [round(gauss_kurtosis, 2),
                                 round(student_kurtosis, 2)],
}))

A Gaussian fit with a freely moving $\sigma(t)$ already absorbs the
volatility history, and it is still the wrong shape: its randomised quantile
residuals {cite:t}`dunn1996randomized` — which are standard normal when the
fitted distribution is right, whatever family produced it — carry an excess
kurtosis of {eval}`gauss_kurtosis:.2f`. Letting the variance move does not
make the tail behave. Adding $\nu(t)$ brings the same diagnostic to
{eval}`student_kurtosis:.2f`.

### Cross-validation risk by iteration count

Boosting a single penalised spline adds flexibility one small step at a time,
so the iteration count is what sets the smoothness of the fitted curves. The
usual way to choose it is cross-validation: refit on resampled folds and take
the count that minimises held-out risk.

Risk decreases across the grid and then flattens, with its minimum at the
upper boundary. The selected iteration count is therefore limited by the
search cap rather than an interior optimum.

In [ ]:
CV_GRID = [[m, m, m] for m in range(100, 4001, 100)]
folds = mb.cv_weights(len(data), type="kfold", B=10, seed=1)
cv = mb.cvrisk_gamboostLSS(FORMS, data=data, families=mb.StudentTLSS(),
                           control=mb.boost_control(4000, 0.1),
                           folds=folds, grid=CV_GRID)

cv_frame = pl.DataFrame({
    "mstop": [pt[0] for pt in cv.grid],
    "cv_risk": np.asarray(cv.aggregate),
})
display(alt.Chart(cv_frame).mark_line(color="#4e79a7").encode(
    x=alt.X("mstop:Q", title="Boosting iterations"),
    y=alt.Y("cv_risk:Q", title="Ten-fold cross-validated risk",
            scale=alt.Scale(zero=False)),
).properties(title="The cross-validated risk flattens without turning up",
             width=560, height=240))

risk = np.asarray(cv.aggregate)
cv_span = float(risk.max() - risk.min())
cv_relative = cv_span / risk.min()
print(f"cross-validated risk falls by {cv_span:.4f} ({cv_relative:.2%}) "
      f"across a {cv_frame['mstop'].max() // cv_frame['mstop'].min():d}-fold "
      f"range of iteration counts; the minimum is at the last grid point, "
      f"{cv.optimal_mstop[0]}")

## Cross-validation risk by `mstop`

Cross-validation evaluates held-out risk over a grid of `mstop` values. The
risk decreases across the grid and then flattens, with its minimum at the
upper boundary. The selected iteration count is limited by the search cap
rather than an interior optimum. Among fits with similar predictive risk,
the iteration count determines how smooth the estimated curves are. This result
does not establish that the Student-t model is well specified.


The whole curve spans {eval}`cv_relative:.2%` of its own level. Cross-validated
risk is essentially flat in the iteration count, which is what a single
well-penalised smooth should produce: past the point where the curve has the
right shape, extra iterations neither help nor hurt held-out prediction much,
and the criterion has nothing left to say.

The three curves compare iteration counts an order of magnitude apart to show
how the fitted structure changes when cross-validation gives little guidance.

In [ ]:
MSTOP = 1000
CANDIDATES = (200, MSTOP, 4000)

curve_grid = pd.DataFrame({"t": np.linspace(data["t"].min(), data["t"].max(), 400)})
curve_grid["ret_pct"] = 0.0
curve_dates = pl.Series(
    "date", pd.Timestamp("2019-01-01") + pd.to_timedelta(
        curve_grid["t"].to_numpy() * 365.25, unit="D"))

PARAM_LABEL = {"mu": "mu(t): drift (% per day)",
               "sigma": "sigma(t): scale (% per day)",
               "df": "nu(t): tail index"}


def read_curves(fit, mstop):
    out = {}
    for name in ("mu", "sigma", "df"):
        kind = "link" if name == "mu" else "response"
        out[name] = np.asarray(fit.predict(parameter=name, type=kind,
                                           newdata=curve_grid, mstop=mstop))
    return out


sensitivity = pl.concat([
    pl.DataFrame({
        "date": curve_dates,
        "value": read_curves(student, m)[name],
        "parameter": [PARAM_LABEL[name]] * len(curve_grid),
        "iterations": [f"{m}"] * len(curve_grid),
    })
    for m in CANDIDATES for name in ("mu", "sigma", "df")
])

alt.Chart(sensitivity).mark_line().encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("value:Q", title=None, scale=alt.Scale(zero=False)),
    color=alt.Color("iterations:N", title="iterations",
                    sort=[str(m) for m in CANDIDATES]),
    row=alt.Row("parameter:N", title=None,
                sort=list(PARAM_LABEL.values())),
).resolve_scale(y="independent").properties(width=520, height=130)

## Analyzing Curve Sensitivity to Model Complexity

This section visualizes how the estimated curves for location ($\mu(t)$), scale ($\sigma(t)$), and tail index ($\nu(t)$) change as the model complexity (number of boosting iterations, $m$) increases.

The goal is to determine if the structural coefficients are stable. The plot shows the curve for the three parameters across a wide range of $m$ values (200 to 4000), allowing us to judge which curves are stable and which are sensitive to the model's flexibility.


The drift and the scale are settled by two hundred iterations; running to four
thousand changes them very little. The tail index is the curve that keeps
moving, because it is the one the data speak to least — every observation
informs the centre and the spread, while only the rare large moves inform the
tail weight. Every number below is read at {eval}`MSTOP:d` iterations, and the
range across this chart shows sensitivity of $\nu(t)$ to the iteration count;
it is not a confidence interval.

## Return-scale quantities from the fitted curves

The fitted curves are converted to quantities on the return scale before they
are compared: the implied standard deviation $\sigma\sqrt{\nu/(\nu-2)}$, the
probability of a move beyond five per cent, and the first-percentile loss.

In [ ]:
fitted = read_curves(student, MSTOP)
mu_t, sigma_t, nu_t = fitted["mu"], fitted["sigma"], fitted["df"]
sd_t = sigma_t * np.sqrt(nu_t / (nu_t - 2))
p_extreme_t = 2 * stats.t.sf(EXTREME / sigma_t, df=nu_t)
var99_t = mu_t + sigma_t * stats.t.ppf(0.01, df=nu_t)

curves = pl.DataFrame({
    "date": curve_dates,
    "implied standard deviation (%)": sd_t,
    "chance of a 5% day": p_extreme_t,
    "first-percentile loss (%)": var99_t,
})
readout = curves.unpivot(index="date", variable_name="quantity",
                         value_name="value")
display(alt.Chart(readout).mark_line(color="#4e79a7").encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("value:Q", title=None, scale=alt.Scale(zero=False)),
    row=alt.Row("quantity:N", title=None),
).resolve_scale(y="independent").properties(width=520, height=130))

print(f"nu(t) runs from {nu_t.min():.2f} to {nu_t.max():.2f}")
print(f"implied standard deviation runs from {sd_t.min():.2f}% to {sd_t.max():.2f}%")
print(f"chance of a {EXTREME:.0f}% day runs from {p_extreme_t.min():.2%} "
      f"to {p_extreme_t.max():.2%}")

## Return-scale metrics

This block converts the abstract coefficients ($\mu, \sigma, \nu$) back into easily interpretable return-scale metrics:
1.  **Implied Standard Deviation ($\sigma_{implied}$):** The scale parameter $\sigma(t)$ is converted to the actual standard deviation using the Student-t formula: $\operatorname{sd} = \sigma\sqrt{\frac{\nu}{\nu-2}}$.
2.  **Extreme Probability ($P_{extreme}$):** Calculates the probability of an event exceeding 5% using the Student-t cumulative distribution function ($\text{sf}$).
3.  **Value at Risk ($\text{VaR}_{99}$):** Estimates the 1st percentile loss, representing the 99% Value at Risk.

This output allows for a direct comparison of the relative risk across the entire sample time series.


The tail index runs from {eval}`nu_t.min():.2f` to {eval}`nu_t.max():.2f`. It
is not a constant of the series, and it is nowhere near the values at which a
Student-t is indistinguishable from a normal law. At the low end the fitted
fourth moment does not exist.

Whether that movement matters is a separate question from whether it is
visible, and it is the question the study set out to answer.

In [ ]:
rank_sd = stats.rankdata(sd_t) / len(sd_t)
rank_p = stats.rankdata(p_extreme_t) / len(p_extreme_t)
spearman_p = float(stats.spearmanr(sd_t, p_extreme_t).statistic)
spearman_var = float(stats.spearmanr(sd_t, var99_t).statistic)
print(f"rank correlation of implied sd with the chance of a {EXTREME:.0f}% day: "
      f"{spearman_p:.3f}")
print(f"rank correlation of implied sd with the first-percentile loss:        "
      f"{spearman_var:.3f}")

# Hold the standard deviation still and see what is left moving.
band = (sd_t > np.quantile(sd_t, 0.4)) & (sd_t < np.quantile(sd_t, 0.6))
band_days = int(band.sum())
band_nu_lo, band_nu_hi = float(nu_t[band].min()), float(nu_t[band].max())
band_p_lo, band_p_hi = float(p_extreme_t[band].min()), float(p_extreme_t[band].max())
band_ratio = band_p_hi / band_p_lo
display(pl.DataFrame({
    "quantity": ["implied standard deviation (%)", "tail index nu",
                 f"chance of a {EXTREME:.0f}% day", "first-percentile loss (%)"],
    "lowest": [f"{sd_t[band].min():.2f}", f"{band_nu_lo:.2f}",
               f"{band_p_lo:.2%}", f"{var99_t[band].min():.2f}"],
    "highest": [f"{sd_t[band].max():.2f}", f"{band_nu_hi:.2f}",
                f"{band_p_hi:.2%}", f"{var99_t[band].max():.2f}"],
}))

## Correlation Analysis Between Risk Measures

This block calculates Spearman's rank correlation coefficients between the three derived metrics across the sample:
1.  **Standard Deviation ($\sigma_{implied}$):** Measures the typical spread.
2.  **Extreme Probability ($P_{extreme}$):** Measures the frequency of large moves.
3.  **First-Percentile Loss ($\text{VaR}_{99}$):** Measures the potential for large losses.

The goal is to determine if volatility alone is a good proxy for the frequency or magnitude of extreme losses.


Across the whole sample the implied standard deviation orders the risk almost
perfectly: its rank correlation with the chance of a five per cent day is
{eval}`spearman_p:.3f`, and with the first-percentile loss
{eval}`spearman_var:.3f`. A practitioner who only ever asks *which periods
were riskier than which* loses very little by looking at volatility alone, and
this study should say so plainly.

The level is another matter. Holding the implied standard deviation inside its
middle quintile — {eval}`band_days:d` days on which the model says the typical
move was the same size — the tail index still runs from
{eval}`band_nu_lo:.2f` to {eval}`band_nu_hi:.2f`, and the chance of a five per
cent day runs from {eval}`band_p_lo:.2%` to {eval}`band_p_hi:.2%`. Same
volatility, {eval}`band_ratio:.1f` times the chance of an extreme day.

### Extreme-day probabilities under a Gaussian model

The comparison uses a Gaussian model with the Student-t fit's time-varying
variance and a different tail shape:
take the implied standard deviation from the fit above, hand it to a normal
law, and ask the same question.

In [ ]:
p_extreme_gauss = 2 * stats.norm.sf(EXTREME / sd_t)
ratio = p_extreme_t / p_extreme_gauss

quantile_rows = []
for q in (0.05, 0.25, 0.50, 0.75, 0.95):
    i = int(np.argmin(np.abs(rank_sd - q)))
    quantile_rows.append({
        "volatility percentile": f"{q:.0%}",
        "implied sd (%)": f"{sd_t[i]:.2f}",
        "tail index nu": f"{nu_t[i]:.2f}",
        "Student-t": f"{p_extreme_t[i]:.3%}",
        "normal, same sd": f"{p_extreme_gauss[i]:.3%}",
        "ratio": f"{ratio[i]:.1f}x",
    })
display(pl.DataFrame(quantile_rows))

ratio_min, ratio_max = float(ratio.min()), float(ratio.max())
ratio_median = float(np.median(ratio))
expected_t = float(p_extreme_t.mean() * n_days)
expected_gauss = float(p_extreme_gauss.mean() * n_days)
print(f"expected number of {EXTREME:.0f}% days over the sample — "
      f"Student-t {expected_t:.0f}, normal with the same variances "
      f"{expected_gauss:.0f}, observed {n_extreme:d}")

## Comparison: Student-t vs. Normal Law

This comparison tests the core hypothesis: Does assuming the wrong distribution (Normal) misspeak the true risk of extreme events?
1.  **Test Setup:** It takes the *correct* time-varying standard deviation ($\sigma_{implied}$) from the Student-t fit and applies it to a Normal distribution.
2.  **Comparison:** It calculates a ratio between the Student-t predicted probability and the Normal predicted probability for various percentiles (5th, 25th, 50th, etc.).
3.  **Conclusion:** The resulting ratio quantifies how many times *more* likely an extreme event is under the Student-t model compared to the Normal model, even when the two models use the same estimated variance.


The normal law with the correct variance is wrong in both directions. In the
calmest periods it understates the chance of a five per cent day by orders of
magnitude, because a normal tail falls away faster than any real return series
does. In the most turbulent it overstates it, because by then the variance it
has been given is so large that five per cent is no longer a remarkable move
for it — while the fitted tail index has risen, making the true distribution
*less* extreme-prone relative to its own spread. Over the sample the ratio
runs from {eval}`ratio_min:.1f` to {eval}`ratio_max:.1f` times, with a median
of {eval}`ratio_median:.1f`.

Counted over the whole history, the Student-t fit expects
{eval}`expected_t:.0f` days beyond five per cent and the normal law with the
same variances expects {eval}`expected_gauss:.0f`, against
{eval}`n_extreme:d` observed. Getting the variance right for every day of the
sample does not get the extreme days right.

## Extreme-day probabilities under Normal and Student-t models

This comparison uses a Normal distribution with the Student-t fit's implied
standard deviation. The Normal model underestimates extreme-loss probabilities
in calm periods and overestimates them in turbulent periods, relative to the
Student-t fit. Matching the variance alone does not match the tail probabilities.


### The distribution over time

The three curves together define a full distribution for every day, which is
the object the study set out to estimate. Its centiles trace how the whole
distribution moved.

In [ ]:
CENTILES = (0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99)
fan = pl.concat([
    pl.DataFrame({
        "date": curve_dates,
        "centile": [f"{q:.0%}"] * len(curve_grid),
        "value": mu_t + sigma_t * stats.t.ppf(q, df=nu_t),
    })
    for q in CENTILES
])

observed = alt.Chart(panel).mark_circle(size=5, opacity=0.25, color="#999").encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("ret_pct:Q", title="Daily total return (%)"),
)
bands = alt.Chart(fan).mark_line(strokeWidth=1.3).encode(
    x="date:T", y="value:Q",
    color=alt.Color("centile:N", title="centile",
                    sort=[f"{q:.0%}" for q in CENTILES],
                    scale=alt.Scale(scheme="blueorange")),
)
(observed + bands).properties(
    title=f"Fitted centiles of the daily return distribution, {security}",
    width=560, height=320)

## Visualizing the Full Distribution (Centiles)

This plot visualizes the full estimated distribution by plotting specific quantiles (percentiles: 1%, 10%, 25%, etc.) for every day.

*   **The lines:** Each colored line represents a different quantile (e.g., the 1st percentile).
*   **The separation:** The distance between these lines ($\text{99th percentile} - \text{1st percentile}$) shows the **width** of the distribution on that day.
*   **Key Insight:** Observing how the outer lines (1% and 99%) pull away from the inner lines (25% and 75%) in turbulent times visually demonstrates the movement of the tail index ($\nu(t)$) in action.


The centiles widen through the first half of 2020 and contract afterwards, and
the outer pair moves by more than the inner pair — which is the tail index at
work. A model with a fixed shape would move every centile by the same factor,
because scaling a distribution moves all of its quantiles together. Here the
one and ninety-nine per cent lines pull away from the twenty-five and
seventy-five per cent lines in the periods when $\nu$ falls, and close back in
when it rises.

## Interpretation of the Full Distribution (Centiles)

This plot visualizes the entire fitted distribution. It shows the empirical daily returns as individual points, and then overlays the calculated percentiles (e.g., the 1st percentile, 99th percentile, etc.) as smooth colored lines.

*   **Movement:** We can observe how the entire distribution shifts and changes shape over time.
*   **Tail Index Evidence:** The key feature is the separation between the outer percentiles (1% and 99%) and the inner ones. When the outer lines pull apart more than the inner lines, it visually confirms that the tail index ($\nu(t)$) is actively changing, making the tails "fatter" or "thinner" independently of the typical movement.


### Quantile residuals and distributional fit

The centiles are only worth reading if the fitted distribution is the right
one. The check is the same randomised quantile residual used earlier: under a
correct fit these are standard normal, and the worm plot shows the deviation
of their ordered values from that reference across the range
{cite:t}`dunn1996randomized`.

In [ ]:
worm = mb.worm_data(student, mstop=MSTOP, random_state=7)
worm_frame = pl.DataFrame(worm) if not isinstance(worm, pl.DataFrame) else worm

qr = np.asarray(mb.quantile_residuals(student, mstop=MSTOP, random_state=7))
print(f"quantile residuals: mean {qr.mean():+.3f}, sd {qr.std():.3f}, "
      f"excess kurtosis {student_kurtosis:+.2f}")
shapiro_p = float(stats.kstest(qr, "norm").pvalue)
print(f"Kolmogorov-Smirnov test against the standard normal: p = {shapiro_p:.3f}")

theoretical = np.sort(stats.norm.ppf(
    (np.arange(len(qr)) + 0.5) / len(qr)))
qq = pl.DataFrame({"theoretical": theoretical, "observed": np.sort(qr)})
line = alt.Chart(pl.DataFrame({"v": [-4.0, 4.0]})).mark_line(
    color="#d62728", strokeDash=[4, 3]).encode(x="v:Q", y="v:Q")
pts = alt.Chart(qq).mark_circle(size=6, opacity=0.4, color="#4e79a7").encode(
    x=alt.X("theoretical:Q", title="Standard normal quantile"),
    y=alt.Y("observed:Q", title="Quantile residual"),
)
(pts + line).properties(
    title="Quantile residuals against the standard normal", width=420, height=320)

## Model Goodness-of-Fit Test (Quantile Residuals)

The worm plot and Kolmogorov-Smirnov test compare the fitted Student-t
quantile residuals with the standard-normal reference. They assess departures
from the fitted distribution; neither establishes that the model is correct.


The residuals sit on the reference line across the whole range, including the
ends where a mis-specified shape shows first, and a Kolmogorov-Smirnov test
against the standard normal returns p = {eval}`shapiro_p:.3f`. The fitted
distribution is not distinguishable from the one that generated the data on
this evidence, which is what licenses reading centiles and tail probabilities
off it.

## Limitations

**One security, one sample.** Every number here describes
{eval}`security` between {eval}`str(start)` and {eval}`str(end)`. A bank's
tail index during a period containing a pandemic and a regional-banking
episode is not a general fact about equities.

**Calendar time is not a mechanism.** The model reads only *when*, so it
describes how the distribution moved without saying why. Attributing the
2020 widening to the pandemic is the reader's inference from the dates, not
the model's finding. A specification with economic covariates — realised
volatility, credit spreads, an option-implied index — would make that
attribution the model's own, and would also be usable for forecasting, which
this one is not.

**Smoothing looks both ways.** $\sigma(t)$ and $\nu(t)$ at any date are
fitted using returns from after that date as well as before, which is
appropriate for describing history and wrong for prediction. Nothing here is
an out-of-sample statement.

**The tail index is the least determined curve.** Only the rare large moves
inform $\nu$, so it carries the most uncertainty of the three, as the
sensitivity to the iteration count shows. The claim it supports — that tail
weight moves independently of scale — survives across that range, but
individual values of $\nu(t)$ should not be read closely.

**No uncertainty bands.** The curves are point estimates. Confidence bands for
a boosted fit need a bootstrap that respects the serial dependence of the
series, which is not run here.

## Study Limitations

This section is critical for interpreting the results. It explicitly outlines the boundaries of the model's claims:
1.  **Scope:** The analysis is limited to one stock and one time period, so the findings cannot be generalized to all equities.
2.  **Causality:** The model only sees *when* a thing happened, not *why*. It describes patterns in the data, but cannot prove economic causation (e.g., it can't say the pandemic *caused* the tail widening).
3.  **Prediction:** Because the parameters are fit over the entire history, the results are descriptive of the past, and thus cannot be directly used for future out-of-sample forecasting.
4.  **Uncertainty:** The current results are point estimates; a full analysis would require complex bootstrapping to provide meaningful confidence bands.


## Summary

Fitting a location, a scale and a tail index jointly as functions of calendar
time separates two things a rolling standard deviation combines. Over
{eval}`security`'s {eval}`n_days:d` trading days the fitted tail index moves
between {eval}`nu_t.min():.2f` and {eval}`nu_t.max():.2f`, independently
enough of the scale that days the model gives the same typical move can differ
by {eval}`band_ratio:.1f` times in their chance of a five per cent move.

For ranking periods by risk this changes little: implied volatility alone
orders the sample at a rank correlation of {eval}`spearman_p:.3f` with the
chance of an extreme day. For pricing an extreme day it changes a great deal.
A normal law handed the correct time-varying variance still misses the
frequency of five per cent moves by a factor running from
{eval}`ratio_min:.1f` to {eval}`ratio_max:.1f` across the sample, and a
Gaussian fit that is free to move its variance leaves an excess kurtosis of
{eval}`gauss_kurtosis:.2f` in residuals that should have none.

The distributional-regression machinery generalises to any question of this
shape. [](motorcycle_variance.ipynb) fits a scale that varies with a predictor
rather than with time, [](choosing_a_distribution.ipynb) treats the family
itself as the question, [](growth_centiles.ipynb) takes the fitted centiles as
the deliverable, and [](sp500_two_lanes.ipynb) asks the same distributional
question of a cross-section rather than one series.

## Overall Summary of Findings

This final summary synthesizes the results across the entire study:
1.  **Separation of Effects:** The primary conclusion is that jointly modeling location ($\mu$), scale ($\sigma$), and tail index ($\nu$) separates what a simple rolling standard deviation combines.
2.  **Impact of Tail:** The tail index's ability to move independently of scale is what allows for a much more accurate picture of extreme risk compared to the Normal model.
3.  **Generalization:** The summary points out that this methodology can be generalized to different types of problems (e.g., modeling cross-sections or different variables), outlining the future directions for the theory.


```{bibliography}
:filter: docname in docnames
```